# Notebook for data analysis and Cleaning

In [26]:
import os
import yaml
from pathlib import Path
from langchain_groq import ChatGroq
from dotenv import load_dotenv
# from langchain_mistralai import MistralAIEmbeddings
import pandas as pd

import numpy as np
import sys
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Load config from YAML file (works in notebook context)
def load_config():
    # Get the notebook directory
    notebook_dir = Path(os.getcwd())
    config_path = notebook_dir.parent / "config" / "config-common-yaml"
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

config = load_config()
load_dotenv()
model_id = "llama-3.1-8b-instant" # 	qwen/qwen3-32b  openai/gpt-oss-120b llama3-8b-8192
groq_api_key = os.getenv("GROQ_API_KEY")

# Configuration from config file
NVIDIA_MODEL = config["NVIDIA_MODEL"]
GROQ_MODEL = config["GROQ_MODEL"]
LLM_TEMPERATURE = config["LLM_TEMPERATURE"]
LLM_MAX_TOKENS = config["LLM_MAX_TOKENS"]
EMBEDDING_MODEL_NAME = config["EMBEDDING_MODEL_NAME"]
NVIDIA_CHAT_TEMPLATE_KWARGS = config["NVIDIA_CHAT_TEMPLATE_KWARGS"]
MIN_CHUNK_LENGTH = config["MIN_CHUNK_LENGTH"]


nvidia_api_key = os.getenv("NVIDIA_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

client = ChatNVIDIA(
        model=NVIDIA_MODEL,
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=nvidia_api_key,
        temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS,
        seed=42,
    )

llm_nvidia = client.bind(**{"chat_template_kwargs": {"enable_thinking": False}, "parallel_tool_calls": True})

llm_groq = ChatGroq(
        model=GROQ_MODEL,
        api_key=groq_api_key,
        temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS,
    )


# Default: Nemotron 3.5 Embed 1B via NVIDIA API (developer-configured)
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
# Create embedding model
# embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
embedding_model = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=os.getenv("NVIDIA_EMBEDDING_API_KEY"),
    truncate="END",
)




/var/folders/0z/6z_9lrtd6j3_gnlsyv9rb5b00000gn/T/ipykernel_51769/346969372.py:39: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  client = ChatNVIDIA(
/Users/rezamohajerpoor/Documents/Documents/Github/FinQA_Agentic_workflow/finqa_app/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3.5-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


## Initial data processing

In [27]:
import pandas as pd
import json

# Read the JSON file
with open('../data/convfinqa_dataset.json', 'r') as f:
    data = json.load(f)

# Convert to DataFrame
df_train = pd.DataFrame(data['train'])
def_dev = pd.DataFrame(data['dev'])
df = pd.concat([df_train, def_dev], ignore_index=True)

# Display the first few rows
print('Shape of the dataframe: ', df.shape)
df.head()

Shape of the dataframe:  (3458, 4)


,id,doc,dialogue,features
0,Single_JKHY/2009/page_28.pdf-3,{'pre_text': '26 | 2009 annual report in fisca...,{'conv_questions': ['what is the net cash from...,"{'num_dialogue_turns': 4, 'has_type2_question'..."
1,Single_RSG/2008/page_114.pdf-2,{'pre_text': 'substantially all of the goodwil...,{'conv_questions': ['what were revenues in 200...,"{'num_dialogue_turns': 4, 'has_type2_question'..."
2,Single_AAPL/2002/page_23.pdf-1,{'pre_text': 'in a new business model such as ...,{'conv_questions': ['what was the total of net...,"{'num_dialogue_turns': 4, 'has_type2_question'..."
3,Single_UPS/2009/page_33.pdf-2,{'pre_text': '( 1 ) includes shares repurchase...,{'conv_questions': ['what was the change in th...,"{'num_dialogue_turns': 6, 'has_type2_question'..."
4,Double_UPS/2009/page_33.pdf,{'pre_text': '( 1 ) includes shares repurchase...,{'conv_questions': ['what was the fluctuation ...,"{'num_dialogue_turns': 7, 'has_type2_question'..."


In [28]:

import json

def _parse(x):
    return json.loads(x) if isinstance(x, str) else x

df["full_example"] = df.apply(
    lambda r: json.dumps(
        {
            "doc": _parse(r["doc"]),
            "dialogue": _parse(r["dialogue"]),
            "features": _parse(r["features"]),
        },
        ensure_ascii=False,
    ),
    axis=1,
)

In [29]:
df.iloc[20]['full_example']

'{"doc": {"pre_text": "value , which may be maturity , the company does not consider these investments to be other-than-temporarily impaired as of december 31 , 2005 and 2004 . gross realized gains and losses for 2005 were $ 15000 and $ 75000 , respectively . gross realized gains and losses for 2004 were $ 628000 and $ 205000 , respectively . gross realized gains for 2003 were $ 1249000 . there were no gross realized losses for 2003 . maturities stated are effective maturities . f . restricted cash at december 31 , 2005 and 2004 , the company held $ 41482000 and $ 49847000 , respectively , in restricted cash . at december 31 , 2005 and 2004 the balance was held in deposit with certain banks predominantly to collateralize conditional stand-by letters of credit in the names of the company\'s landlords pursuant to certain operating lease agreements . g . property and equipment property and equipment consist of the following at december 31 ( in thousands ) : depreciation expense for the ye

## Summarize each document 

In [33]:
from joblib import Parallel, delayed
import math
import time



def summarize_doc(doc,llm):

    response = llm.invoke(f"""
            System: You are a helpful assistant that writes a topic for a financial question scenario, together with a summary of the problem and key concepts covered by the scenario.
            This extracted information helps in building a knowledge base for financial question answering, in a way that the approrpriate examples will be extracted from the knowledge base based on the similarity of the query to the summary that you are writing on this exmaple.
            \n
            
            User: Financial question scenario:\n
            
            {doc}.

            Please keep the description concise and to the point (up to 200 words).

            Please return the following:
            
            Topic: 

            Summary Description:

            Key concepts covered by this example:
            """)
    return str(response.content)



def summarize_doc_with_retry(doc, llm, max_retries=3, retry_delay=5):
    attempt = 0
    while True:
        try:
            return summarize_doc(doc, llm)
        except Exception as e:
            attempt += 1
            if attempt > max_retries:
                raise
            print(f"  Error: {e}. Retrying in {retry_delay}s (attempt {attempt}/{max_retries})...")
            time.sleep(retry_delay)

# for parallel processing need to use another technology like joblib 

In [32]:
doc_samples = df.iloc[200]['full_example']

summary_groq = summarize_doc(doc_samples,llm_groq)
summary_nvidia = summarize_doc(doc_samples,llm_nvidia)

print(f"Summary from Groq LLM:\n{summary_groq}\n")
print(f"Summary from Nvidia LLM:\n{summary_nvidia}\n")

Summary from Groq LLM:
**Topic:**  
Year‑over‑Year Segment Sales Change and Percentage Calculation  

**Summary Description:**  
The example presents a financial excerpt detailing segment sales and earnings for a metal‑beverage‑packaging business across 2008‑2010. Using the reported net sales figures, a dialogue asks for (1) the sales difference between 2009 and 2008, (2) the sales decrease from 2009 to 2010, (3) the net change (difference of the two differences), and (4) the net change expressed as a proportion of the 2009‑2008 difference. The solution demonstrates extracting numeric values from the text/table, performing subtraction, and then dividing to obtain a percentage (≈ 67.5 %). This illustrates how to translate narrative financial data into quantitative comparisons and ratio calculations.

**Key concepts covered by this example:**  
- Reading and locating segment sales numbers in narrative and tabular financial disclosures.  
- Computing year‑over‑year differences (subtractio

In [34]:
batch_size = 20  # You can adjust this value
num_batches = math.ceil(len(df) / batch_size)

summaries = []
for i in range(num_batches):
    batch = df.iloc[i*batch_size : (i+1)*batch_size]
    batch_summaries = Parallel(n_jobs=-1, backend="threading")(delayed(summarize_doc_with_retry)(doc, llm_nvidia) for doc in batch['full_example'])
    summaries.extend(batch_summaries)
    print(f'Batch {i+1} of {num_batches} done.')
    time.sleep(5)  # To avoid hitting rate limits


df['summary'] = summaries

Batch 1 of 173 done.
Batch 2 of 173 done.
Batch 3 of 173 done.
Batch 4 of 173 done.
Batch 5 of 173 done.
Batch 6 of 173 done.
Batch 7 of 173 done.
Batch 8 of 173 done.
Batch 9 of 173 done.
  Error: [502] Bad Gateway
Upstream request failed.. Retrying in 5s (attempt 1/3)...
Batch 10 of 173 done.
Batch 11 of 173 done.
Batch 12 of 173 done.
Batch 13 of 173 done.
Batch 14 of 173 done.
Batch 15 of 173 done.
Batch 16 of 173 done.
Batch 17 of 173 done.
Batch 18 of 173 done.
Batch 19 of 173 done.
Batch 20 of 173 done.
Batch 21 of 173 done.
Batch 22 of 173 done.
Batch 23 of 173 done.
Batch 24 of 173 done.
Batch 25 of 173 done.
Batch 26 of 173 done.
Batch 27 of 173 done.
Batch 28 of 173 done.
Batch 29 of 173 done.
Batch 30 of 173 done.
Batch 31 of 173 done.
Batch 32 of 173 done.
Batch 33 of 173 done.
Batch 34 of 173 done.
Batch 35 of 173 done.
Batch 36 of 173 done.
Batch 37 of 173 done.
Batch 38 of 173 done.
Batch 39 of 173 done.
Batch 40 of 173 done.
Batch 41 of 173 done.
Batch 42 of 173 done.


In [36]:
print('Shape of the dataframe: ', df.shape)
print(f'Length of summaries: {len(summaries)}')

Shape of the dataframe:  (3458, 6)
Length of summaries: 3458


In [37]:
df.to_csv('../data/convfinqa_wsummary.csv', index=False)

## Add embeddings

In [38]:
# Load the DataFrame from the CSV file
df_s = pd.read_csv('../data/convfinqa_wsummary.csv')
df_s.head()

,id,doc,dialogue,features,full_example,summary
0,Single_JKHY/2009/page_28.pdf-3,{'pre_text': '26 | 2009 annual report in fisca...,{'conv_questions': ['what is the net cash from...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""26 | 2009 annual report ...",Topic: Analysis of Net Cash from Operating Act...
1,Single_RSG/2008/page_114.pdf-2,{'pre_text': 'substantially all of the goodwil...,{'conv_questions': ['what were revenues in 200...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""substantially all of the...",Topic: Pro Forma Financial Analysis and Revenu...
2,Single_AAPL/2002/page_23.pdf-1,{'pre_text': 'in a new business model such as ...,{'conv_questions': ['what was the total of net...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""in a new business model ...",Topic: Gross Margin and Net Sales Trend Analys...
3,Single_UPS/2009/page_33.pdf-2,{'pre_text': '( 1 ) includes shares repurchase...,{'conv_questions': ['what was the change in th...,"{'num_dialogue_turns': 6, 'has_type2_question'...","{""doc"": {""pre_text"": ""( 1 ) includes shares re...",Topic: Cumulative Total Shareowner Return Comp...
4,Double_UPS/2009/page_33.pdf,{'pre_text': '( 1 ) includes shares repurchase...,{'conv_questions': ['what was the fluctuation ...,"{'num_dialogue_turns': 7, 'has_type2_question'...","{""doc"": {""pre_text"": ""( 1 ) includes shares re...",Topic: Cumulative Total Shareowner Return Comp...


In [93]:
# create embedding for the summaries with retry mechanism
import time

def embed_with_retry(text, max_retries=3, delay=5):
    """Embed text with retry on 502 Bad Gateway errors."""
    for attempt in range(max_retries):
        try:
            return embedding_model.embed_query(text)
        except Exception as e:
            if "502" in str(e) or "Bad Gateway" in str(e):
                if attempt < max_retries - 1:
                    print(f"  ⚠️  502 Bad Gateway on attempt {attempt + 1}, retrying in {delay}s...")
                    time.sleep(delay)
                    continue
            raise


batch_size = 20  # You can adjust this value
num_batches = math.ceil(len(df_s) / batch_size)

embeddings = []
for i in range(num_batches):
    batch = df.iloc[i*batch_size : (i+1)*batch_size]
    batch_embeddings = Parallel(n_jobs=-1, backend="threading")(delayed(embed_with_retry)(text) for text in batch['summary'])
    embeddings.extend(batch_embeddings)
    print(f'Batch {i+1} of {num_batches} done.')
    

df_s["summary_emb"] = embeddings

# df_s["summary_emb"] = df_s["summary"].apply(lambda x: embed_with_retry(x))



Batch 1 of 173 done.
Batch 2 of 173 done.
Batch 3 of 173 done.
Batch 4 of 173 done.
Batch 5 of 173 done.
Batch 6 of 173 done.
Batch 7 of 173 done.
Batch 8 of 173 done.
Batch 9 of 173 done.
Batch 10 of 173 done.
Batch 11 of 173 done.
Batch 12 of 173 done.
Batch 13 of 173 done.
Batch 14 of 173 done.
Batch 15 of 173 done.
Batch 16 of 173 done.
Batch 17 of 173 done.
Batch 18 of 173 done.
Batch 19 of 173 done.
Batch 20 of 173 done.
Batch 21 of 173 done.
Batch 22 of 173 done.
Batch 23 of 173 done.
  ⚠️  502 Bad Gateway on attempt 1, retrying in 5s...
Batch 24 of 173 done.
Batch 25 of 173 done.
Batch 26 of 173 done.
Batch 27 of 173 done.
Batch 28 of 173 done.
Batch 29 of 173 done.
Batch 30 of 173 done.
Batch 31 of 173 done.
Batch 32 of 173 done.
Batch 33 of 173 done.
Batch 34 of 173 done.
Batch 35 of 173 done.
Batch 36 of 173 done.
Batch 37 of 173 done.
Batch 38 of 173 done.
Batch 39 of 173 done.
Batch 40 of 173 done.
Batch 41 of 173 done.
Batch 42 of 173 done.
Batch 43 of 173 done.
Batch 44 

In [94]:
# simple search through the data
row = df_s[df_s['features'].apply(lambda x: eval(str(x))['has_type2_question'])].iloc[30]

print('summary: ', row['summary'])
print('Features: ', row['features'])
print('Dialogue: ', row['dialogue'])

summary:  Topic: Analysis of U.S. Pharmaceutical Sales Return, Rebate, and Discount Liability Roll-Forward

Summary Description: This scenario examines the year-over-year changes in U.S. pharmaceutical sales return, rebate, and discount liability balances and the associated reduction of net sales from 2017 to 2018. It involves calculating the absolute change in liability balances and net sales reductions, followed by determining these changes as percentages of the 2017 baseline values.

Key concepts covered by this example:
- Liability roll-forward analysis
- Year-over-year financial variance calculation
- Percentage change computation
- Sales return, rebate, and discount accounting
- Impact of sales deductions on net revenue
Features:  {'num_dialogue_turns': 4, 'has_type2_question': True, 'has_duplicate_columns': False, 'has_non_numeric_values': False}
Dialogue:  {'conv_questions': ['what was the change in the total of u.s . pharmaceutical sales return, rebate, and discount liability 

In [95]:
df_s.to_csv('../data/convfinqa_wembed.csv', index=False)
df_s.head()

,id,doc,dialogue,features,full_example,summary,summary_emb,dialogue_dict,conv_questions,conv_answers,turn_program,executed_answers,qa_split,conv_text_full,features_dict,num_dialogue_turns,has_type2_question,has_duplicate_columns,has_non_numeric_values
0,Single_JKHY/2009/page_28.pdf-3,{'pre_text': '26 | 2009 annual report in fisca...,{'conv_questions': ['what is the net cash from...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""26 | 2009 annual report ...",Topic: Analysis of Net Cash from Operating Act...,"[0.039764404, 0.026657104, -0.00868988, -0.001...",{'conv_questions': ['what is the net cash from...,[what is the net cash from operating activitie...,"[206588, 181001, 25587, 14.1%]","[206588, 181001, subtract(206588, 181001), sub...","[206588.0, 181001.0, 25587.0, 0.14136]","[False, False, False, False]","""""""\nQ: what is the net cash from operating ac...","{'num_dialogue_turns': 4, 'has_type2_question'...",4,False,False,False
1,Single_RSG/2008/page_114.pdf-2,{'pre_text': 'substantially all of the goodwil...,{'conv_questions': ['what were revenues in 200...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""substantially all of the...",Topic: Pro Forma Financial Analysis and Revenu...,"[0.05432129, 0.0005335808, -0.031219482, -0.01...",{'conv_questions': ['what were revenues in 200...,"[what were revenues in 2008?, what were they i...","[9362.2, 9244.9, 117.3, 1.3%]","[9362.2, 9244.9, subtract(9362.2, 9244.9), sub...","[9362.2, 9244.9, 117.3, 0.01269]","[False, False, False, False]","""""""\nQ: what were revenues in 2008?\nReason: 9...","{'num_dialogue_turns': 4, 'has_type2_question'...",4,False,False,False
2,Single_AAPL/2002/page_23.pdf-1,{'pre_text': 'in a new business model such as ...,{'conv_questions': ['what was the total of net...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""in a new business model ...",Topic: Gross Margin and Net Sales Trend Analys...,"[0.04043579, 0.03967285, -0.029312134, -0.0264...",{'conv_questions': ['what was the total of net...,"[what was the total of net sales in 2001?, and...","[5363, 7983, -2620, -32%]","[5363, 7983, subtract(5363, 7983), subtract(53...","[5363.0, 7983.0, -2620.0, -0.3282]","[False, False, False, False]","""""""\nQ: what was the total of net sales in 200...","{'num_dialogue_turns': 4, 'has_type2_question'...",4,False,False,False
3,Single_UPS/2009/page_33.pdf-2,{'pre_text': '( 1 ) includes shares repurchase...,{'conv_questions': ['what was the change in th...,"{'num_dialogue_turns': 6, 'has_type2_question'...","{""doc"": {""pre_text"": ""( 1 ) includes shares re...",Topic: Cumulative Total Shareowner Return Comp...,"[0.045440674, 0.037109375, -0.019836426, -0.00...",{'conv_questions': ['what was the change in th...,[what was the change in the performance of the...,"[-24.05, -24.05%, 102.11, 2.11, 2.11%, -26.16%]","[subtract(75.95, const_100), subtract(75.95, c...","[-24.05, -0.2405, 102.11, 2.11, 0.0211, -0.2616]","[False, False, False, False, False, False]","""""""\nQ: what was the change in the performance...","{'num_dialogue_turns': 6, 'has_type2_question'...",6,False,False,False
4,Double_UPS/2009/page_33.pdf,{'pre_text': '( 1 ) includes shares repurchase...,{'conv_questions': ['what was the fluctuation ...,"{'num_dialogue_turns': 7, 'has_type2_question'...","{""doc"": {""pre_text"": ""( 1 ) includes shares re...",Topic: Cumulative Total Shareowner Return Comp...,"[0.036376953, 0.039886475, -0.00843811, -0.005...",{'conv_questions': ['what was the fluctuation ...,[what was the fluctuation of the performance p...,"[-8.94, -8.9%, -24.05, -24.05%, 2.11, 2.11%, -...","[subtract(91.06, const_100), subtract(91.06, c...","[-8.94, -0.0894, -24.05, -0.2405, 2.11, 0.0211...","[False, False, True, True, True, True, True]","""""""\nQ: what was the fluctuation of the perfor...","{'num_dialogue_turns': 7, 'has_type2_question'...",7,True,False,False


## Search relevant summaries as a function



In [109]:
# Load the DataFrame from the CSV file
df_f = pd.read_csv('../data/convfinqa_wembed.csv')

In [110]:
print(type(df_f["summary_emb"].iloc[20]))
print(df_f["summary_emb"].iloc[20][0:60], '...') 
print('Need to parse the embeddings from string to list of floats')

<class 'str'>
[0.049621582, -0.01159668, 0.0047912598, -0.0024776459, 0.03 ...
Need to parse the embeddings from string to list of floats


In [111]:
import re
import numpy as np

def parse_embedding(s):
    if isinstance(s, np.ndarray):
        return s
    if isinstance(s, list):
        return np.array(s, dtype=float)
    if not isinstance(s, str):
        raise ValueError(f"Unexpected embedding type: {type(s)}")

    # Remove brackets and extra spaces
    s = s.strip().replace("[", "").replace("]", "")
    # Split on commas
    nums = re.split(r",", s.strip())
    # Convert to float
    return np.array([float(x) for x in nums if x != ""], dtype=float)

df_f["summary_emb"] = df_f["summary_emb"].apply(parse_embedding)


In [112]:
print(type(df_f["summary_emb"].iloc[20]))
print(df_f["summary_emb"].iloc[20][0:60], '...') 

<class 'numpy.ndarray'>
[ 0.04962158 -0.01159668  0.00479126 -0.00247765  0.03372192 -0.00171757
  0.00975037 -0.02796936 -0.00678635 -0.01434326  0.01126099 -0.03231812
 -0.007061    0.00451279 -0.02203369 -0.00687027  0.0380249   0.01204681
 -0.01982117  0.00183201  0.0029583   0.00952911  0.01660156  0.00534821
 -0.01939392  0.04882812  0.00709534  0.00748825 -0.00135231 -0.0001421
 -0.03451538 -0.00030565  0.00195312  0.00158215 -0.04031372 -0.02056885
 -0.01625061  0.01280212  0.01074982 -0.0137558   0.01225281  0.00800323
 -0.04602051  0.00777817  0.0136795  -0.00160217  0.00432587 -0.00068092
 -0.05084229  0.03564453 -0.00130749  0.01673889 -0.00887299  0.02363586
  0.02809143  0.00552368  0.01383209  0.04052734 -0.03979492 -0.02116394] ...


In [113]:
import numpy as np
from numpy.linalg import norm



def find_relevant_document_fast(embedding_model, query, df, embed_col, top_k=3):
    query_emb = np.array(embedding_model.embed_query(query), dtype=float)

    def cosine(x):
        if x is None:
            return -1.0
        nq = norm(query_emb)
        nx = norm(x)
        if nq == 0 or nx == 0:
            return -1.0
        return float(np.dot(x, query_emb) / (nx * nq))

    sims = df[embed_col].apply(cosine)
    valid = sims > -1.0
    top_docs = df.loc[valid].iloc[np.argsort(sims[valid])[-top_k:][::-1]]
    return top_docs

In [114]:
query = "What was S&P 500's stock price trend in the last quarter in 2012?"
relevant_docs = find_relevant_document_fast(embedding_model, query, df_f, 'summary_emb', top_k=3)
for i, row in relevant_docs.iterrows():
    print(f"Document {i}:")
    print("Summary:", row['summary'])
    print("Dialogue:", row['dialogue'])
    print("Features:", row['features'])
    print("\n---\n")    

Document 2377:
Summary: Topic: Share Repurchase Program Analysis

Summary Description: This scenario analyzes a company's share repurchase activity during the quarter ended December 31, 2012. It involves calculating the portion of total stock purchases made in October 2012, determining the number of shares purchased between November 26 and December 31, 2012, and computing the total value of those purchases at the average price paid per share, both in absolute terms and in millions.

Key concepts covered by this example:
- Share repurchases
- Average price paid per share
- Total value of stock purchases
- Percentage of total purchases
- Financial table data extraction and calculation
Dialogue: {'conv_questions': ['what portion of total stock purchases occurred in october 2012?', 'how many shares were purchased during november 26 - december 31, 2012?', 'at that average price?', 'what is the total value of these purchases?', 'what about in millions?'], 'conv_answers': ['27.1%', '1395288',

## Restructure the FinQA dataset

In [117]:
import ast

# Extract the dictionary from the string in df_e.iloc[0]['dialogue']
features_str = df_f.iloc[0]['features']
features_dict = ast.literal_eval(features_str)
features_dict_keys = ast.literal_eval(df_f.iloc[0]['features']).keys()
print(features_dict_keys)
print(type(features_dict))  # Should be <class 'dict'>
print(features_dict)

dict_keys(['num_dialogue_turns', 'has_type2_question', 'has_duplicate_columns', 'has_non_numeric_values'])
<class 'dict'>
{'num_dialogue_turns': 4, 'has_type2_question': False, 'has_duplicate_columns': False, 'has_non_numeric_values': False}


In [118]:


# For each key in the first dictionary, create a new column
for key in features_dict_keys:
    df_f[key] = df_f['features'].apply(lambda x: ast.literal_eval(x)).apply(lambda d: d[key])

df_f.head()

,id,doc,dialogue,features,full_example,summary,summary_emb,dialogue_dict,conv_questions,conv_answers,turn_program,executed_answers,qa_split,conv_text_full,features_dict,num_dialogue_turns,has_type2_question,has_duplicate_columns,has_non_numeric_values
0,Single_JKHY/2009/page_28.pdf-3,{'pre_text': '26 | 2009 annual report in fisca...,{'conv_questions': ['what is the net cash from...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""26 | 2009 annual report ...",Topic: Analysis of Net Cash from Operating Act...,"[0.039764404, 0.026657104, -0.00868988, -0.001...",{'conv_questions': ['what is the net cash from...,['what is the net cash from operating activiti...,"['206588', '181001', '25587', '14.1%']","['206588', '181001', 'subtract(206588, 181001)...","[206588.0, 181001.0, 25587.0, 0.14136]","[False, False, False, False]","""""""\nQ: what is the net cash from operating ac...","{'num_dialogue_turns': 4, 'has_type2_question'...",4,False,False,False
1,Single_RSG/2008/page_114.pdf-2,{'pre_text': 'substantially all of the goodwil...,{'conv_questions': ['what were revenues in 200...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""substantially all of the...",Topic: Pro Forma Financial Analysis and Revenu...,"[0.05432129, 0.0005335808, -0.031219482, -0.01...",{'conv_questions': ['what were revenues in 200...,"['what were revenues in 2008?', 'what were the...","['9362.2', '9244.9', '117.3', '1.3%']","['9362.2', '9244.9', 'subtract(9362.2, 9244.9)...","[9362.2, 9244.9, 117.3, 0.01269]","[False, False, False, False]","""""""\nQ: what were revenues in 2008?\nReason: 9...","{'num_dialogue_turns': 4, 'has_type2_question'...",4,False,False,False
2,Single_AAPL/2002/page_23.pdf-1,{'pre_text': 'in a new business model such as ...,{'conv_questions': ['what was the total of net...,"{'num_dialogue_turns': 4, 'has_type2_question'...","{""doc"": {""pre_text"": ""in a new business model ...",Topic: Gross Margin and Net Sales Trend Analys...,"[0.04043579, 0.03967285, -0.029312134, -0.0264...",{'conv_questions': ['what was the total of net...,"['what was the total of net sales in 2001?', '...","['5363', '7983', '-2620', '-32%']","['5363', '7983', 'subtract(5363, 7983)', 'subt...","[5363.0, 7983.0, -2620.0, -0.3282]","[False, False, False, False]","""""""\nQ: what was the total of net sales in 200...","{'num_dialogue_turns': 4, 'has_type2_question'...",4,False,False,False
3,Single_UPS/2009/page_33.pdf-2,{'pre_text': '( 1 ) includes shares repurchase...,{'conv_questions': ['what was the change in th...,"{'num_dialogue_turns': 6, 'has_type2_question'...","{""doc"": {""pre_text"": ""( 1 ) includes shares re...",Topic: Cumulative Total Shareowner Return Comp...,"[0.045440674, 0.037109375, -0.019836426, -0.00...",{'conv_questions': ['what was the change in th...,['what was the change in the performance of th...,"['-24.05', '-24.05%', '102.11', '2.11', '2.11%...","['subtract(75.95, const_100)', 'subtract(75.95...","[-24.05, -0.2405, 102.11, 2.11, 0.0211, -0.2616]","[False, False, False, False, False, False]","""""""\nQ: what was the change in the performance...","{'num_dialogue_turns': 6, 'has_type2_question'...",6,False,False,False
4,Double_UPS/2009/page_33.pdf,{'pre_text': '( 1 ) includes shares repurchase...,{'conv_questions': ['what was the fluctuation ...,"{'num_dialogue_turns': 7, 'has_type2_question'...","{""doc"": {""pre_text"": ""( 1 ) includes shares re...",Topic: Cumulative Total Shareowner Return Comp...,"[0.036376953, 0.039886475, -0.00843811, -0.005...",{'conv_questions': ['what was the fluctuation ...,['what was the fluctuation of the performance ...,"['-8.94', '-8.9%', '-24.05', '-24.05%', '2.11'...","['subtract(91.06, const_100)', 'subtract(91.06...","[-8.94, -0.0894, -24.05, -0.2405, 2.11, 0.0211...","[False, False, True, True, True, True, True]","""""""\nQ: what was the fluctuation of the perfor...","{'num_dialogue_turns': 7, 'has_type2_question'...",7,True,False,False


In [119]:
df_f.to_csv('../data/convfinqa_full.csv', index=False)